<a href="https://colab.research.google.com/github/wappynerd/Estimador-de-volatilidad-con-RF/blob/main/pairs_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [103]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from datetime import datetime, timedelta
from itertools import combinations

# --- LIBRERÍAS DE INTERFAZ PROFESIONAL ---
from rich.console import Console
from rich.table import Table
from rich.text import Text
import warnings

# Silenciar warnings de yfinance/pandas para mantener la consola limpia
warnings.filterwarnings('ignore')

console = Console()

# --- FUNCIONES TÉCNICAS ---
def calcular_hurst(ts):
    lags = range(2, 20)
    tau = [np.sqrt(np.std(np.subtract(ts[lag:], ts[:-lag]))) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0] * 2.0

def calcular_half_life(residuos):
    res_lag = residuos.shift(1).fillna(method='bfill')
    delta_res = residuos - res_lag
    res_lag_const = sm.add_constant(res_lag)
    modelo = sm.OLS(delta_res, res_lag_const).fit()
    lambda_val = modelo.params.iloc[1]
    if lambda_val >= 0: return 999
    return round(-np.log(2) / lambda_val, 1)

# --- MOTOR DE BÚSQUEDA Y VISUALIZACIÓN ---
def buscar_cointegracion_por_sectores(sectores_dict):
    fin = datetime.now()
    inicio = fin - timedelta(days=150)

    console.print(f"\n[bold cyan]INICIANDO ESCÁNER DE COINTEGRACIÓN[/bold cyan] | Período: {inicio.strftime('%Y-%m-%d')} a {fin.strftime('%Y-%m-%d')}")
    console.print("="*80)

    for sector, lista_tickers in sectores_dict.items():
        tickers_clean = [t.split(" - ")[0].strip() + ".BA" for t in lista_tickers]

        # Barra de estado mientras descarga datos
        with console.status(f"[bold yellow]Descargando y analizando: {sector.upper()}...", spinner="dots"):
            try:
                df_precios = yf.download(tickers_clean, start=inicio, end=fin, progress=False)['Close']
                df_precios = df_precios.ffill().dropna()
            except Exception as e:
                console.print(f"[bold red]❌ Error en {sector}: {e}[/bold red]")
                continue

            pares_resultados = []
            activos = df_precios.columns.tolist()

            for t1, t2 in combinations(activos, 2):
                Y = df_precios[t1]
                X = sm.add_constant(df_precios[t2])
                modelo = sm.OLS(Y, X).fit()
                residuos = modelo.resid
                p_valor = adfuller(residuos)[1]

                if p_valor < 0.05:  # Umbral de confianza
                    hl = calcular_half_life(residuos)
                    hurst = round(calcular_hurst(residuos.values), 2)
                    res_centrados = residuos - residuos.mean()
                    cruces = ((res_centrados.shift(1) * res_centrados) < 0).sum()
                    z_score = (residuos.iloc[-1] - residuos.mean()) / residuos.std()

                    p1_name = t1.replace('.BA','')
                    p2_name = t2.replace('.BA','')

                    # Lógica de señales con TICKERS EXPLÍCITOS
                    if z_score >= 2.0:
                        accion = f"VENDER {p1_name} / COMPRAR {p2_name}"
                        color_accion = "bold red"
                    elif z_score <= -2.0:
                        accion = f"COMPRAR {p1_name} / VENDER {p2_name}"
                        color_accion = "bold green"
                    else:
                        accion = "Neutral"
                        color_accion = "dim"

                    pares_resultados.append({
                        'P1': p1_name, 'P2': p2_name,
                        'P-Val': p_valor,
                        'Z': z_score,
                        'Hurst': hurst,
                        'H-Life': hl,
                        'Cruces': cruces,
                        'Acción': accion,
                        'Color': color_accion
                    })

        # --- DIBUJAR LA TABLA PRO ---
        if pares_resultados:
            # Ordenar por Hurst (menor a mayor, buscando mayor reversión)
            pares_resultados.sort(key=lambda x: x['Hurst'])

            # Tabla con títulos en BLANCO
            table = Table(title=f"📊 SECTOR: {sector.upper()}", title_style="bold white", show_header=True, header_style="bold white on #333333")

            table.add_column("Par (P1 / P2)", style="cyan", justify="left")
            table.add_column("P-Value", justify="right")
            table.add_column("Z-Score", justify="right")
            table.add_column("Hurst", justify="right")
            table.add_column("H-Life", justify="right")
            table.add_column("Cruces", justify="center")
            table.add_column("Acción Sugerida", justify="center")

            for row in pares_resultados:
                # Formateo numérico
                pval_str = f"{row['P-Val']:.4f}"

                # Z-Score con color según desviación
                z_val = row['Z']
                if z_val >= 2.0: z_str = f"[bold red]{z_val:.2f}[/bold red]"
                elif z_val <= -2.0: z_str = f"[bold green]{z_val:.2f}[/bold green]"
                else: z_str = f"{z_val:.2f}"

                # Hurst visual (resalta si es < 0.4 indicando buena reversión)
                h_val = row['Hurst']
                h_str = f"[bold green]{h_val:.2f}[/bold green]" if h_val < 0.4 else f"{h_val:.2f}"

                table.add_row(
                    f"{row['P1']} / {row['P2']}",
                    pval_str,
                    z_str,
                    h_str,
                    str(row['H-Life']),
                    str(row['Cruces']),
                    f"[{row['Color']}]{row['Acción']}[/{row['Color']}]"
                )

            console.print(table)
            console.print("") # Espacio visual entre sectores
        else:
            console.print(f"[dim italic]No se detectó cointegración significativa (<0.05) en {sector}.[/dim italic]\n")

# --- CONFIGURACIÓN DE CARTERA ---
sectores = {
    "Bancos y Sistema Financiero": ["GGAL - 24hs", "BMA - 24hs", "BBAR - 24hs", "SUPV - 24hs", "VALO - 24hs", "BPAT - 24hs", "BHIP - 24hs"],
    "Energía y Utilities": ["YPFD - 24hs", "PAMP - 24hs", "CEPU - 24hs", "EDN - 24hs", "TRAN - 24hs", "TGSU2 - 24hs", "TGNO4 - 24hs", "METR - 24hs"],
    "Materiales e Industria": ["ALUA - 24hs", "TXAR - 24hs", "LOMA - 24hs", "MIRG - 24hs"],
    "Real Estate y Otros": ["CRES - 24hs", "IRSA - 24hs", "COME - 24hs", "TECO2 - 24hs", "BYMA - 24hs"]
}

if __name__ == "__main__":
    buscar_cointegracion_por_sectores(sectores)

INICIANDO ESCÁNER DE COINTEGRACIÓN | Período: 2025-11-18 a 2026-04-17

================================================================================

Output()

                     📊 SECTOR: BANCOS Y SISTEMA FINANCIERO                      
┏━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Par (P1 / P2) ┃ P-Value ┃ Z-Score ┃ Hurst ┃ H-Life ┃ Cruces ┃ Acción Sugerida ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ BHIP / BPAT   │  0.0368 │   -0.19 │  0.17 │    4.5 │   15   │     Neutral     │
│ BPAT / SUPV   │  0.0030 │   -0.83 │  0.20 │    3.9 │   20   │     Neutral     │
│ BPAT / GGAL   │  0.0101 │   -1.25 │  0.24 │    5.1 │   22   │     Neutral     │
│ BBAR / BMA    │  0.0442 │    0.43 │  0.31 │    4.2 │   16   │     Neutral     │
└───────────────┴─────────┴─────────┴───────┴────────┴────────┴─────────────────┘

Output()

                         📊 SECTOR: ENERGÍA Y UTILITIES                          
┏━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Par (P1 / P2) ┃ P-Value ┃ Z-Score ┃ Hurst ┃ H-Life ┃ Cruces ┃ Acción Sugerida ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ CEPU / TGNO4  │  0.0138 │   -0.88 │  0.10 │    4.3 │   14   │     Neutral     │
│ EDN / PAMP    │  0.0007 │    0.37 │  0.10 │    2.3 │   26   │     Neutral     │
│ EDN / METR    │  0.0457 │   -0.34 │  0.19 │    4.4 │   15   │     Neutral     │
│ CEPU / METR   │  0.0005 │   -0.41 │  0.21 │    4.4 │   12   │     Neutral     │
│ CEPU / PAMP   │  0.0145 │   -0.12 │  0.31 │    5.6 │   10   │     Neutral     │
│ TRAN / YPFD   │  0.0117 │    0.07 │  0.38 │    6.3 │   15   │     Neutral     │
│ CEPU / EDN    │  0.0164 │   -0.43 │  0.39 │    5.9 │   18   │     Neutral     │
└───────────────┴─────────┴─────────┴───────┴────────┴────────┴─────────────────┘

Output()

                        📊 SECTOR: MATERIALES E INDUSTRIA                        
┏━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Par (P1 / P2) ┃ P-Value ┃ Z-Score ┃ Hurst ┃ H-Life ┃ Cruces ┃ Acción Sugerida ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ ALUA / TXAR   │  0.0035 │   -0.73 │  0.24 │    4.9 │   12   │     Neutral     │
│ ALUA / LOMA   │  0.0023 │    0.15 │  0.28 │    4.4 │   12   │     Neutral     │
│ LOMA / TXAR   │  0.0399 │   -0.77 │  0.36 │    6.3 │   16   │     Neutral     │
│ ALUA / MIRG   │  0.0408 │    1.01 │  0.38 │    8.9 │   12   │     Neutral     │
└───────────────┴─────────┴─────────┴───────┴────────┴────────┴─────────────────┘

Output()

                         📊 SECTOR: REAL ESTATE Y OTROS                          
┏━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Par (P1 / P2) ┃ P-Value ┃ Z-Score ┃ Hurst ┃ H-Life ┃ Cruces ┃ Acción Sugerida ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ CRES / IRSA   │  0.0004 │   -1.56 │  0.42 │    5.1 │   18   │     Neutral     │
└───────────────┴─────────┴─────────┴───────┴────────┴────────┴─────────────────┘

In [104]:
import yfinance as yf
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

# --- CONFIGURACIÓN ---
UMBRAL_Z = 2
TICKER1 = "BBAR.BA"
TICKER2 = "BPAT.BA"

def visualizar_par_neon(t1, t2):
    try:
        # 1. Descarga de datos
        fin = datetime.now()
        inicio = fin - timedelta(days=150)
        df = yf.download([t1, t2], start=inicio, end=fin, progress=False)['Close'].dropna().ffill()

        # 2. Cálculos Estadísticos
        Y = df[t1]
        X = sm.add_constant(df[t2])
        modelo = sm.OLS(Y, X).fit()
        spread = modelo.resid
        std_dev = spread.std()
        media = spread.mean()
        z_score = (spread - media) / std_dev
        z_actual = z_score.iloc[-1]

        # 3. Colores Fluo (Paleta Cyberpunk/Bloomberg)
        BG_COLOR = '#000000'
        GRID_COLOR = '#222222'
        CYAN_FLUO = '#00FFFF'   # Ticker 1
        MAGENTA_FLUO = '#FF00FF' # Ticker 2
        LIME_FLUO = '#39FF14'    # Compra / Z-Score
        RED_FLUO = '#FF3131'     # Venta

        # 4. Construcción del Tablero
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
            subplot_titles=(f'<span style="color:white">DINÁMICA DE PRECIOS: {t1} vs {t2}</span>',
                            f'<span style="color:white">Z-SCORE DEL SPREAD (Actual: {z_actual:.2f})</span>')
        )

        # --- FILA 1: PRECIOS ---
        fig.add_trace(go.Scatter(x=df.index, y=df[t1], name=t1,
                                 line=dict(color=CYAN_FLUO, width=2)), row=1, col=1, secondary_y=False)
        fig.add_trace(go.Scatter(x=df.index, y=df[t2], name=t2,
                                 line=dict(color=MAGENTA_FLUO, width=2)), row=1, col=1, secondary_y=True)

        # --- FILA 2: Z-SCORE ---
        fig.add_trace(go.Scatter(x=z_score.index, y=z_score, name='Z-Score',
                                 line=dict(color=LIME_FLUO, width=2),
                                 fill='tozeroy', fillcolor='rgba(57, 255, 20, 0.05)'), row=2, col=1)

        # Referencias de Trading Fluo
        fig.add_hline(y=0, line_dash="solid", line_color="#444", row=2, col=1)
        fig.add_hline(y=UMBRAL_Z, line_dash="dash", line_color=RED_FLUO, line_width=2,
                      annotation_text="VENDER (+2σ)", annotation_font_color=RED_FLUO, row=2, col=1)
        fig.add_hline(y=-UMBRAL_Z, line_dash="dash", line_color=LIME_FLUO, line_width=2,
                      annotation_text="COMPRAR (-2σ)", annotation_font_color=LIME_FLUO, row=2, col=1)

        # Estética General (Dark Mode Neon)
        fig.update_layout(
            height=850,
            paper_bgcolor=BG_COLOR,
            plot_bgcolor=BG_COLOR,
            title=dict(
                text=f"<b style='color:white'>Terminal Pairs Trading</b><br><span style='color:{CYAN_FLUO}'>By Cointegration</span>",
                x=0.05, y=0.95, font=dict(family="Courier New, monospace", size=24)
            ),
            hovermode="x unified",
            hoverlabel=dict(bgcolor="#111", font_size=12, font_family="Courier New", font_color="white"),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(color="white")),
            margin=dict(l=50, r=50, t=130, b=50)
        )

        # Formato de Ejes (Gris muy oscuro)
        fig.update_xaxes(showgrid=True, gridcolor=GRID_COLOR, zeroline=False, tickfont=dict(color='white'))
        fig.update_yaxes(showgrid=True, gridcolor=GRID_COLOR, zeroline=False, tickfont=dict(color='white'))

        fig.update_yaxes(title_text=f"Price {t1}", row=1, col=1, secondary_y=False, title_font_color=CYAN_FLUO)
        fig.update_yaxes(title_text=f"Price {t2}", row=1, col=1, secondary_y=True, title_font_color=MAGENTA_FLUO)
        fig.update_yaxes(title_text="Z-Score", row=2, col=1, title_font_color=LIME_FLUO)

        fig.show()

    except Exception as e:
        print(f"Error en el sistema: {e}")

# Ejecutar
visualizar_par_neon(TICKER1, TICKER2)